# 03 — Distribution Shift Analysis
**Project:** ViT Reliability & Explainability Under Medical Distribution Shift
**Author:** Sosna Worku

**Goal:** Evaluate both models on CheXpert (OOD). Measure AUC drop, calibration degradation, and MC-Dropout uncertainty.

---
**Research Questions answered here:**
- RQ1: How much does performance drop under distribution shift?
- RQ2: Does uncertainty estimation flag OOD failures?

Run all cells top to bottom every new Colab session.

## 0. Setup — Run every session

In [ ]:
import os, sys

REPO_PATH = '/content/vit-medical-shift'
GITHUB    = 'https://github.com/sossyh/vit-medical-shift.git'

if os.path.exists(REPO_PATH):
    os.system(f'git -C {REPO_PATH} pull origin main')
else:
    os.system(f'git clone {GITHUB} {REPO_PATH}')

sys.path.insert(0, REPO_PATH)
print('Repo ready!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
!pip install -q timm torchmetrics grad-cam einops pyyaml
print('Packages ready!')

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Paths & Device

In [ ]:
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()

# NIH paths (in-distribution)
NIH_CSV   = '/content/drive/MyDrive/data/nih/Data_Entry_2017.csv'
NIH_IMGS  = '/content/drive/MyDrive/data/nih/images'

# CheXpert paths (OOD) — update after downloading
CHEX_CSV  = '/content/drive/MyDrive/data/chexpert/CheXpert-v1.0-small/valid.csv'
CHEX_IMGS = '/content/drive/MyDrive/data/chexpert/CheXpert-v1.0-small'

# Checkpoints
VIT_CKPT    = '/content/drive/MyDrive/checkpoints/vit_nih/best.pth'
RESNET_CKPT = '/content/drive/MyDrive/checkpoints/resnet_nih/best.pth'

print('ViT checkpoint   :', os.path.exists(VIT_CKPT))
print('ResNet checkpoint:', os.path.exists(RESNET_CKPT))
print('CheXpert CSV     :', os.path.exists(CHEX_CSV))

## 2. Copy NIH Images to Local Disk

In [ ]:
import shutil

LOCAL_NIH = '/content/nih_images'

if os.path.exists(LOCAL_NIH) and len(os.listdir(LOCAL_NIH)) > 1000:
    print(f'NIH images already local: {len(os.listdir(LOCAL_NIH)):,}')
else:
    print('Copying NIH images to local disk...')
    os.makedirs(LOCAL_NIH, exist_ok=True)
    os.system(f'rsync -a {NIH_IMGS}/ {LOCAL_NIH}/')
    print(f'Done! {len(os.listdir(LOCAL_NIH)):,} images')

NIH_IMGS = LOCAL_NIH
print('NIH IMG_DIR:', NIH_IMGS)

## 3. Load Models from Checkpoints

In [ ]:
from src.model import get_vit, get_resnet

# Load ViT
vit_model = get_vit(num_classes=14, pretrained=False).to(device)
vit_model.load_state_dict(torch.load(VIT_CKPT, map_location=device))
vit_model.eval()
print('ViT loaded!')

# Load ResNet
resnet_model = get_resnet(num_classes=14, pretrained=False).to(device)
resnet_model.load_state_dict(torch.load(RESNET_CKPT, map_location=device))
resnet_model.eval()
print('ResNet loaded!')

## 4. In-Distribution Evaluation (NIH validation set)

In [ ]:
import torch.nn as nn
from src.dataset import get_nih_loaders
from src.evaluate import compute_auc, compute_ece, print_metrics
from src.train import validate

criterion = nn.BCEWithLogitsLoss()

_, val_loader_id = get_nih_loaders(
    csv_path    = NIH_CSV,
    img_dir     = NIH_IMGS,
    batch_size  = 32,
    subset      = 0.2,
    num_workers = 0
)

print('--- ViT ID ---')
_, vit_logits_id, vit_labels_id = validate(vit_model, val_loader_id, criterion, device)
vit_auc_id  = compute_auc(vit_logits_id, vit_labels_id)
vit_ece_id  = compute_ece(vit_logits_id, vit_labels_id)
print_metrics(vit_auc_id, vit_ece_id, 'ViT ID')

print('\n--- ResNet ID ---')
_, res_logits_id, res_labels_id = validate(resnet_model, val_loader_id, criterion, device)
res_auc_id  = compute_auc(res_logits_id, res_labels_id)
res_ece_id  = compute_ece(res_logits_id, res_labels_id)
print_metrics(res_auc_id, res_ece_id, 'ResNet ID')

## 5. OOD Evaluation (CheXpert)
Run this cell only after CheXpert is downloaded to Drive.

In [ ]:
from src.dataset import CheXpertDataset, get_transforms
from torch.utils.data import DataLoader

# Load CheXpert validation set
chex_dataset = CheXpertDataset(
    csv_path  = CHEX_CSV,
    img_root  = CHEX_IMGS,
    transform = get_transforms('val')
)
chex_loader = DataLoader(
    chex_dataset,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 0
)
print(f'CheXpert samples: {len(chex_dataset):,}')

print('\n--- ViT OOD ---')
_, vit_logits_ood, vit_labels_ood = validate(vit_model, chex_loader, criterion, device)
vit_auc_ood  = compute_auc(vit_logits_ood, vit_labels_ood)
vit_ece_ood  = compute_ece(vit_logits_ood, vit_labels_ood)
print_metrics(vit_auc_ood, vit_ece_ood, 'ViT OOD')

print('\n--- ResNet OOD ---')
_, res_logits_ood, res_labels_ood = validate(resnet_model, chex_loader, criterion, device)
res_auc_ood  = compute_auc(res_logits_ood, res_labels_ood)
res_ece_ood  = compute_ece(res_logits_ood, res_labels_ood)
print_metrics(res_auc_ood, res_ece_ood, 'ResNet OOD')

## 6. Distribution Shift Analysis — AUC Drop

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.utils import NIH_LABELS

# Build comparison table
shift_df = pd.DataFrame({
    'Label'         : NIH_LABELS + ['Mean'],
    'ViT ID'        : [vit_auc_id[l]  for l in NIH_LABELS] + [vit_auc_id['mean']],
    'ViT OOD'       : [vit_auc_ood[l] for l in NIH_LABELS] + [vit_auc_ood['mean']],
    'ResNet ID'     : [res_auc_id[l]  for l in NIH_LABELS] + [res_auc_id['mean']],
    'ResNet OOD'    : [res_auc_ood[l] for l in NIH_LABELS] + [res_auc_ood['mean']],
})
shift_df['ViT Drop']    = shift_df['ViT ID']    - shift_df['ViT OOD']
shift_df['ResNet Drop'] = shift_df['ResNet ID'] - shift_df['ResNet OOD']
print(shift_df.to_string(index=False))

# Plot AUC drop comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(NIH_LABELS))
w = 0.35

# ID vs OOD per model
axes[0].bar([i-w/2 for i in x], [vit_auc_id[l]  for l in NIH_LABELS], w,
            label='ViT ID',     color='#378ADD', alpha=0.9)
axes[0].bar([i+w/2 for i in x], [vit_auc_ood[l] for l in NIH_LABELS], w,
            label='ViT OOD',    color='#378ADD', alpha=0.4)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(NIH_LABELS, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('AUC')
axes[0].set_title('ViT: ID vs OOD AUC')
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')

axes[1].bar([i-w/2 for i in x], [res_auc_id[l]  for l in NIH_LABELS], w,
            label='ResNet ID',  color='#E05C2A', alpha=0.9)
axes[1].bar([i+w/2 for i in x], [res_auc_ood[l] for l in NIH_LABELS], w,
            label='ResNet OOD', color='#E05C2A', alpha=0.4)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(NIH_LABELS, rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('AUC')
axes[1].set_title('ResNet: ID vs OOD AUC')
axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
os.makedirs(f'{REPO_PATH}/results/figures', exist_ok=True)
plt.savefig(f'{REPO_PATH}/results/figures/distribution_shift_auc.png', dpi=150)
plt.show()
print('Saved!')

## 7. Calibration Analysis — ECE ID vs OOD

In [ ]:
from src.evaluate import plot_reliability_diagram, temperature_scale

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

titles = [
    ('ViT — In Distribution (NIH)',     vit_logits_id,  vit_labels_id),
    ('ViT — Out of Distribution (CheXpert)', vit_logits_ood, vit_labels_ood),
    ('ResNet — In Distribution (NIH)',   res_logits_id,  res_labels_id),
    ('ResNet — Out of Distribution (CheXpert)', res_logits_ood, res_labels_ood),
]

for ax, (title, logits, labels) in zip(axes.flatten(), titles):
    import numpy as np
    probs  = torch.sigmoid(logits).numpy().flatten()
    labels_np = labels.numpy().flatten()
    bins   = np.linspace(0, 1, 11)
    centers, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        centers.append((lo+hi)/2)
        accs.append(labels_np[mask].mean() if mask.sum() > 0 else 0)
    ece = compute_ece(logits, labels)
    ax.bar(centers, accs, width=0.09, alpha=0.7, color='#378ADD')
    ax.plot([0,1],[0,1],'k--', lw=1.5, label='Perfect')
    ax.set_title(f'{title}\nECE={ece:.4f}', fontsize=9)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{REPO_PATH}/results/figures/calibration_id_vs_ood.png', dpi=150)
plt.show()
print('Saved!')

## 8. MC-Dropout Uncertainty Analysis

In [ ]:
from src.model import mc_dropout_predict
import numpy as np

# Get a batch from each domain
id_images,  id_labels,  _ = next(iter(val_loader_id))
ood_images, ood_labels, _ = next(iter(chex_loader))

# MC-Dropout predictions (20 forward passes)
vit_mean_id,  vit_std_id  = mc_dropout_predict(vit_model, id_images,  n_passes=20, device=device)
vit_mean_ood, vit_std_ood = mc_dropout_predict(vit_model, ood_images, n_passes=20, device=device)

print('ViT Uncertainty (std) ID  :', vit_std_id.mean().item())
print('ViT Uncertainty (std) OOD :', vit_std_ood.mean().item())
print('\nHigher uncertainty on OOD = model knows it is uncertain = good!')

# Plot uncertainty distributions
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(vit_std_id.numpy().flatten(),  bins=50, alpha=0.6,
        color='#378ADD', label=f'ID (NIH) mean={vit_std_id.mean():.4f}')
ax.hist(vit_std_ood.numpy().flatten(), bins=50, alpha=0.6,
        color='#E05C2A', label=f'OOD (CheXpert) mean={vit_std_ood.mean():.4f}')
ax.set_xlabel('Prediction Uncertainty (std across 20 passes)')
ax.set_ylabel('Count')
ax.set_title('ViT MC-Dropout Uncertainty: ID vs OOD', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{REPO_PATH}/results/figures/mc_dropout_uncertainty.png', dpi=150)
plt.show()
print('Saved!')

## 9. Save Results & Download

In [ ]:
import json
from google.colab import files

os.makedirs(f'{REPO_PATH}/results/metrics', exist_ok=True)

# Save shift metrics
shift_metrics = {
    'vit'   : {'id_auc': vit_auc_id,  'ood_auc': vit_auc_ood,
               'id_ece': vit_ece_id,  'ood_ece': vit_ece_ood},
    'resnet': {'id_auc': res_auc_id,  'ood_auc': res_auc_ood,
               'id_ece': res_ece_id,  'ood_ece': res_ece_ood},
}
with open(f'{REPO_PATH}/results/metrics/shift_metrics.json', 'w') as f:
    json.dump(shift_metrics, f, indent=2)

shift_df.to_csv(f'{REPO_PATH}/results/metrics/shift_analysis.csv', index=False)
print('Metrics saved! Downloading...')

files.download(f'{REPO_PATH}/results/metrics/shift_metrics.json')
files.download(f'{REPO_PATH}/results/metrics/shift_analysis.csv')
files.download(f'{REPO_PATH}/results/figures/distribution_shift_auc.png')
files.download(f'{REPO_PATH}/results/figures/calibration_id_vs_ood.png')
files.download(f'{REPO_PATH}/results/figures/mc_dropout_uncertainty.png')
print('Done!')